# Price Elasticity of Demand
### Analysing Beef Market Price-Quantity Relationships

## 1. Project Overview
This notebook analyses quarterly beef market data to compute and visualise price elasticity of demand. We explore how quantity demanded responds to price changes over time and estimate elasticity coefficients using log-log regression.

## 2. Learning Objectives
- Understand the economic concept of price elasticity of demand
- Calculate point and arc elasticity from time-series data
- Apply log-log regression to estimate elasticity coefficients
- Visualise demand curves and elasticity trends
- Interpret elasticity values in business context

## 3. Business / Research Problem
**Question:** How responsive is beef demand to price changes? Is beef demand elastic (|E| > 1) or inelastic (|E| < 1), and how has this changed over time?

## 4. Why This Analysis Matters
Price elasticity is a fundamental concept in economics and business strategy. Understanding demand sensitivity helps producers set prices, governments plan agricultural policy, and analysts forecast market responses to supply shocks.

## 5. Dataset Overview
The dataset contains quarterly beef market observations:
- `Year` — observation year
- `Quarter` — Q1 through Q4
- `Quantity` — quantity of beef demanded
- `Price` — price per unit

## 6. Dataset Source and License Notes
- **Source:** Beef market data (economic dataset)
- **File:** `beef.csv`
- **License:** Public / educational use

## 7. Environment Setup

In [1]:
import subprocess, sys
for p in ['pandas','numpy','matplotlib','seaborn','scipy','scikit-learn']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

Ready.


## 8. Imports

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from pathlib import Path
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

Imports OK.


## 9. Configuration / Constants

In [3]:
CSV_FILE = Path('beef.csv')

## 10. Dataset Loading

In [4]:
df = pd.read_csv(CSV_FILE)
print(f'Shape: {df.shape}')
df.head(10)

Shape: (91, 4)


,Year,Quarter,Quantity,Price
0,1977,1,22.9976,142.1667
1,1977,2,22.6131,143.9333
2,1977,3,23.4054,146.5000
3,1977,4,22.7401,150.8000
4,1978,1,22.0441,160.0000
5,1978,2,21.7602,182.5333
6,1978,3,21.6064,186.2000
7,1978,4,21.8814,186.4333
8,1979,1,20.5086,211.7000
9,1979,2,19.0408,231.5000


## 11. Data Validation Checks

In [5]:
print('Columns:', df.columns.tolist())
print(f'\nMissing values: {df.isnull().sum().sum()}')
print(f'Year range: {df["Year"].min()} — {df["Year"].max()}')
print(f'\nBasic stats:')
df.describe()

Columns: ['Year', 'Quarter', 'Quantity', 'Price']

Missing values: 0
Year range: 1977 — 1999

Basic stats:


,Year,Quarter,Quantity,Price
count,91.000000,91.000000,91.000000,91.000000
mean,1987.879121,2.483516,18.403309,250.440293
std,6.604435,1.119153,1.813343,37.010231
min,1977.000000,1.000000,15.891500,142.166700
25%,1982.000000,1.500000,17.043950,231.333350
50%,1988.000000,2.000000,18.167800,250.100000
75%,1993.500000,3.000000,19.358650,280.716700
max,1999.000000,4.000000,23.405400,300.400000


## 12. Data Cleaning
1. Create a time index from Year and Quarter
2. Sort chronologically
3. Compute period-over-period changes

In [6]:
df['Period'] = df['Year'].astype(str) + '-Q' + df['Quarter'].astype(str)
df = df.sort_values(['Year','Quarter']).reset_index(drop=True)
df['Price_pct_change'] = df['Price'].pct_change()
df['Qty_pct_change'] = df['Quantity'].pct_change()
print(f'Records: {len(df)}')
df.head()

Records: 91


,Year,Quarter,Quantity,Price,Period,Price_pct_change,Qty_pct_change
0,1977,1,22.9976,142.1667,1977-Q1,NaN,NaN
1,1977,2,22.6131,143.9333,1977-Q2,0.012426,-0.016719
2,1977,3,23.4054,146.5000,1977-Q3,0.017833,0.035037
3,1977,4,22.7401,150.8000,1977-Q4,0.029352,-0.028425
4,1978,1,22.0441,160.0000,1978-Q1,0.061008,-0.030607


## 13. Exploratory Data Analysis

In [7]:
print('Price statistics:')
print(df['Price'].describe())
print(f'\nQuantity statistics:')
print(df['Quantity'].describe())
print(f'\nCorrelation (Price vs Quantity): {df["Price"].corr(df["Quantity"]):.3f}')

Price statistics:
count     91.000000
mean     250.440293
std       37.010231
min      142.166700
25%      231.333350
50%      250.100000
75%      280.716700
max      300.400000
Name: Price, dtype: float64

Quantity statistics:
count    91.000000
mean     18.403309
std       1.813343
min      15.891500
25%      17.043950
50%      18.167800
75%      19.358650
max      23.405400
Name: Quantity, dtype: float64

Correlation (Price vs Quantity): -0.949


## 14. Univariate Analysis

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['Price'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price')

axes[1].hist(df['Quantity'], bins=20, color='coral', edgecolor='white')
axes[1].set_title('Quantity Distribution')
axes[1].set_xlabel('Quantity')
plt.tight_layout(); plt.show()

## 15. Bivariate / Multivariate Analysis
### Demand Curve Visualisation

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Demand curve (Price vs Quantity)
axes[0].scatter(df['Quantity'], df['Price'], alpha=0.6, color='steelblue')
z = np.polyfit(df['Quantity'], df['Price'], 1)
p_fit = np.poly1d(z)
q_range = np.linspace(df['Quantity'].min(), df['Quantity'].max(), 100)
axes[0].plot(q_range, p_fit(q_range), 'r--', label=f'Linear fit')
axes[0].set_xlabel('Quantity')
axes[0].set_ylabel('Price')
axes[0].set_title('Demand Curve')
axes[0].legend()

# Time series
axes[1].plot(df.index, df['Price'], label='Price', color='steelblue')
ax2 = axes[1].twinx()
ax2.plot(df.index, df['Quantity'], label='Quantity', color='coral')
axes[1].set_title('Price and Quantity Over Time')
axes[1].set_xlabel('Period')
axes[1].set_ylabel('Price', color='steelblue')
ax2.set_ylabel('Quantity', color='coral')
axes[1].legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout(); plt.show()

## 16. Feature-Specific Insights
### Computing Price Elasticity of Demand
$$E_d = \frac{\%\Delta Q}{\%\Delta P}$$

In [10]:
# Arc elasticity for each period
df['arc_elasticity'] = df['Qty_pct_change'] / df['Price_pct_change']
df_elas = df.dropna(subset=['arc_elasticity']).copy()
df_elas = df_elas[np.isfinite(df_elas['arc_elasticity'])]

print(f'Mean elasticity: {df_elas["arc_elasticity"].mean():.3f}')
print(f'Median elasticity: {df_elas["arc_elasticity"].median():.3f}')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_elas.index, df_elas['arc_elasticity'], marker='o', markersize=3, color='steelblue')
ax.axhline(y=-1, color='red', linestyle='--', label='Unit elastic (E=-1)')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.set_title('Arc Price Elasticity Over Time')
ax.set_ylabel('Elasticity')
ax.set_xlabel('Period Index')
ax.legend()
plt.tight_layout(); plt.show()

Mean elasticity: 0.482
Median elasticity: -0.340


## 17. Statistical Checks / Hypothesis Testing
### Log-Log Regression for Elasticity Estimation
$$\ln(Q) = \alpha + \beta \cdot \ln(P) + \epsilon$$
The coefficient $\beta$ estimates the constant elasticity.

In [11]:
df['log_price'] = np.log(df['Price'])
df['log_qty'] = np.log(df['Quantity'])

X = df[['log_price']].values
y = df['log_qty'].values
model = LinearRegression().fit(X, y)

print(f'Log-log regression results:')
print(f'  Elasticity (β): {model.coef_[0]:.4f}')
print(f'  Intercept (α):  {model.intercept_:.4f}')
print(f'  R²:             {model.score(X, y):.4f}')

# Significance test
slope, intercept, r_value, p_value, std_err = stats.linregress(df['log_price'], df['log_qty'])
print(f'  p-value:        {p_value:.4e}')
print(f'  Significant:    {p_value < 0.05}')

Log-log regression results:
  Elasticity (β): -0.5314
  Intercept (α):  5.8364
  R²:             0.8521
  p-value:        1.0689e-38
  Significant:    True


## 18. Visual Analysis
### Log-Log Demand Curve and Elasticity Distribution

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log-log scatter with regression line
axes[0].scatter(df['log_price'], df['log_qty'], alpha=0.6, color='steelblue')
x_fit = np.linspace(df['log_price'].min(), df['log_price'].max(), 100)
axes[0].plot(x_fit, model.predict(x_fit.reshape(-1,1)), 'r--', linewidth=2)
axes[0].set_xlabel('ln(Price)')
axes[0].set_ylabel('ln(Quantity)')
axes[0].set_title(f'Log-Log Demand Curve (β={model.coef_[0]:.3f})')

# Elasticity distribution
axes[1].hist(df_elas['arc_elasticity'].clip(-5, 5), bins=30, color='seagreen', edgecolor='white')
axes[1].axvline(x=-1, color='red', linestyle='--', label='Unit elastic')
axes[1].set_title('Distribution of Arc Elasticity')
axes[1].set_xlabel('Elasticity')
axes[1].legend()

plt.tight_layout(); plt.show()

## 19. Key Findings
1. **Beef demand is inelastic** — the elasticity coefficient is between 0 and -1.
2. **Price and quantity show a negative relationship** as expected by demand theory.
3. **Log-log regression** provides a clean constant-elasticity estimate.
4. **Elasticity varies over time** — some quarters show elastic responses.
5. **The demand curve slopes downward** confirming standard economic theory.

## 20. Limitations
- Only price and quantity are available — no income, substitute prices, or seasonal controls.
- Arc elasticity is noisy for small percentage changes.
- Endogeneity: price and quantity are simultaneously determined.
- No instrument variables to address supply-side identification.

## 21. Recommendations / Next Steps
1. Include income and substitute good prices for a fuller demand model.
2. Apply 2SLS (instrumental variables) to address endogeneity.
3. Test for structural breaks in elasticity over time.
4. Compare beef elasticity with other proteins (chicken, pork).
5. Apply seasonal decomposition to isolate trend elasticity.

## 22. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Computing elasticity as ΔQ/ΔP | Ignores base levels | Use percentage changes |
| Assuming constant elasticity | Elasticity varies along the curve | Estimate by segment or log-log |
| Ignoring simultaneous equation bias | Price is endogenous | Use IV/2SLS |
| Dividing by zero % change | Produces infinity | Filter or use arc mid-point formula |
| Interpreting positive elasticity | Violates demand law | Check data or model specification |

## 23. Mini Challenge / Exercises
1. **Mid-point formula:** Recompute arc elasticity using the mid-point method — does it reduce noise?
2. **Rolling elasticity:** Compute a 4-quarter rolling elasticity — how does it trend?
3. **Forecast:** Use the log-log model to predict quantity if price increases by 10%.
4. **Cross-price:** If you had chicken price data, how would you estimate cross-price elasticity?
5. **Segmentation:** Split the data into pre/post midpoint year and compare elasticities.

## 24. Final Summary / Key Takeaways
```
TAKEAWAY 1  Beef demand is price-inelastic — quantity declines less than proportionally to price increases.
TAKEAWAY 2  Log-log regression provides the cleanest elasticity estimate.
TAKEAWAY 3  Arc elasticity is noisy but reveals time-varying demand sensitivity.
TAKEAWAY 4  The standard downward-sloping demand curve holds for this market.
TAKEAWAY 5  Additional variables (income, substitutes) are needed for a complete demand model.
```